In [ ]:
import json
import pandas as pd
import os

In [ ]:
# Load the JSONL files line by line
business_data = []
with open('yelp/yelp_academic_dataset_business.json', 'r') as f:
	for line in f:
		business_data.append(json.loads(line.strip()))
business = pd.DataFrame(business_data)

review_data = []
with open('yelp/yelp_academic_dataset_review.json', 'r') as f:
	for line in f:
		review_data.append(json.loads(line.strip()))
review = pd.DataFrame(review_data)

In [ ]:
import pandas as pd

def clean_business_data(df):
    df = df.copy()
    
    # Mark validity of each field
    df['_valid_name'] = df['name'].apply(lambda x: pd.notna(x) and str(x).strip() != '')
    df['_valid_review'] = df['review_count'].apply(lambda x: pd.notna(x) and (x > 0))
    df['_valid_categories'] = df['categories'].apply(lambda x: pd.notna(x) and str(x).strip() != '')
    
    # Comprehensive validity marker
    df['_is_valid'] = df['_valid_name'] & df['_valid_review'] & df['_valid_categories']
    
    # Separate valid and invalid data
    clean_df = df[df['_is_valid']].drop(['_valid_name', '_valid_review', '_valid_categories', '_is_valid'], axis=1)
    invalid_df = df[~df['_is_valid']].drop(['_valid_name', '_valid_review', '_valid_categories', '_is_valid'], axis=1)
    
    # Generate error report
    error_report = {
        'total_original': len(df),
        'total_cleaned': len(clean_df),
        'total_removed': len(invalid_df),
        'name_errors': len(df[~df['_valid_name']]),
        'review_errors': len(df[~df['_valid_review']]),
        'categories_errors': len(df[~df['_valid_categories']]),
        'sample_invalid': invalid_df.head(3).to_dict('records')
    }
    
    return clean_df, error_report

# Usage example
cleaned_business, report = clean_business_data(business)

# Print error report
print("=== Data Cleaning Report ===")
print(f"Original data count: {report['total_original']}")
print(f"Cleaned data count: {report['total_cleaned']}")
print(f"Removed records: {report['total_removed']}")
print("\n--- Invalid Record Distribution ---")
print(f"Invalid names: {report['name_errors']}")
print(f"Invalid review counts: {report['review_errors']}")
print(f"Invalid categories: {report['categories_errors']}")

print("\n--- Sample Invalid Records ---")
for i, rec in enumerate(report['sample_invalid'], 1):
    print(f"\nExample {i}:")
    print(f"ID: {rec.get('business_id', 'N/A')}")
    print(f"Name: {rec.get('name', 'N/A')}")
    print(f"Review count: {rec.get('review_count', 'N/A')}")
    print(f"Categories: {rec.get('categories', 'N/A')}")

In [ ]:
# Create a dictionary to store business information
business_info = {}

# Extract relevant fields from the business dataframe
for index, row in cleaned_business.iterrows():
    business_id = row['business_id']
    
    # Extract necessary fields (handle potential missing values)
    name = row.get('name', 'Unknown')
    stars = row.get('stars', 0)
    review_count = row.get('review_count', 0)
    categories = row.get('categories', '')
    
    # Process categories - split by comma and limit to 10
    if categories:
        category_list = [cat.strip() for cat in categories.split(',')]
        if len(category_list) > 10:
            category_list = category_list[:10]
        categories_text = ', '.join(category_list)
    else:
        categories_text = ''
    
    # Create a descriptive sentence
    description = f"{name}, which is a business with {stars} stars rating based on {review_count} reviews."
    if categories_text:
        description += f" It is categorized as {categories_text}."
    
    # Store in dictionary
    business_info[business_id] = {
        'name': name,
        'stars': stars,
        'review_count': review_count,
        'categories': categories_text,
        'description': description
    }

# Print a sample of the dictionary
sample_business = list(business_info.items())[:3]
for business_id, info in sample_business:
    print(f"Business ID: {business_id}")
    print(f"Description: {info['description']}")
    print("-" * 50)

In [ ]:
# Get IDs from restaurant_business (which is filtered business)
bussiness_ids = set(cleaned_business['business_id'])
cleaned_reviews = review[review['business_id'].isin(bussiness_ids)]

# Check the size of the filtered reviews
print(f"Total reviews in dataset: {len(review)}")
print(f"Restaurant reviews: {len(cleaned_reviews)}")
print(f"Percentage of reviews : {len(cleaned_reviews) / len(review) * 100:.2f}%")

In [ ]:
# Count how many reviews each user has
user_review_counts = cleaned_reviews['user_id'].value_counts()

# Filter to get users with more than 25 reviews
active_users = user_review_counts[user_review_counts >= 25]

# Display the statistics
print(f"Number of users with more than 25 reviews: {len(active_users)}")

In [ ]:
review_gt_9 = cleaned_reviews[cleaned_reviews['user_id'].isin(active_users.index)]
print(f"Total reviews from users with more than 20 reviews: {len(review_gt_9)}")

In [ ]:
# Group reviews by user and get the last 75 reviews for users with more than 75 reviews
# First convert the date column to datetime for proper sorting
cleaned_reviews.loc[:, 'date'] = pd.to_datetime(cleaned_reviews['date'])

# Create a function to get the last 75 reviews for each user
def get_last_75_reviews(user_reviews):
    if len(user_reviews) > 75:
        return user_reviews.sort_values('date').tail(75)
    else:
        return user_reviews

# Apply the function to each user group
last_75_reviews_per_user = review_gt_9.groupby('user_id').apply(get_last_75_reviews)

# Reset index to flatten the DataFrame
last_75_reviews_per_user = last_75_reviews_per_user.reset_index(drop=True)

# Count how many users have more than 75 reviews
user_review_counts = review_gt_9['user_id'].value_counts()  # 确保这个变量已定义
users_with_more_than_75 = user_review_counts[user_review_counts > 75]
print(f"Number of users with more than 75 reviews: {len(users_with_more_than_75)}")

# Count total reviews in the new dataset
print(f"Total reviews in filtered dataset: {len(last_75_reviews_per_user)}")

# Check how many reviews were reduced
reduction = len(review_gt_9) - len(last_75_reviews_per_user)
percentage = (reduction / len(review_gt_9)) * 100
print(f"Reduced dataset by {reduction:,} reviews ({percentage:.2f}% reduction)")

In [ ]:
last_75_reviews_per_user.groupby('user_id').size().describe()

In [ ]:
# Calculate statistics: number of items, interactions, and sparsity

# Calculate basic statistics
n_users = last_75_reviews_per_user['user_id'].nunique()
n_items = last_75_reviews_per_user['business_id'].nunique()
n_interactions = len(last_75_reviews_per_user)

# Calculate sparsity
total_possible_interactions = n_users * n_items
sparsity = 1 - (n_interactions / total_possible_interactions)

print(f"Dataset Statistics:")
print(f"Number of users: {n_users:,}")
print(f"Number of items (restaurants): {n_items:,}")
print(f"Number of interactions: {n_interactions:,}")
print(f"Total possible interactions: {total_possible_interactions:,}")
print(f"Sparsity: {sparsity:.6f} ({sparsity*100:.4f}%)")

# Calculate average interactions per user
avg_interactions_per_user = n_interactions / n_users
print(f"Average interactions per user: {avg_interactions_per_user:.2f}")

# Calculate average interactions per item
avg_interactions_per_item = n_interactions / n_items
print(f"Average interactions per item: {avg_interactions_per_item:.2f}")


In [ ]:
# Create a dataset by combining business descriptions and reviews
def create_combined_dataset(last_75_reviews_per_user, business_info):
    # Initialize lists to hold our data
    data = []
    
    # Iterate through reviews
    for idx, review in last_75_reviews_per_user.iterrows():
        business_id = review['business_id']
        
        # Check if we have information about this business
        if business_id in business_info:
            # Get business description
            business_desc = business_info[business_id]['description']
            business_name = business_info[business_id]['name']
            
            # Get review text
            review_text = review['text']
            
            # Get star rating
            star_rating = review['stars']
            
            # Create an entry with business description as context and review as response
            entry = {
                'business_description': business_desc,
                #'review_text': review_text,
                'rating': star_rating,
                'business_id': business_id,
                'UserID': review['user_id'],
                'bussiness_name': business_name,
                #'review_id': review['review_id'],
                'date': review['date']
            }
            
            data.append(entry)
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Check how many entries we have
    print(f"Created dataset with {len(df):,} entries")
    
    return df

# Create the dataset
combined_dataset = create_combined_dataset(last_75_reviews_per_user, business_info)

In [ ]:
combined_dataset

In [ ]:
# Create a function to split the reviews into training and validation sets
def create_train_valid_split(df, valid_percentage=0.2):
    """
    Split the reviews for each user into training and validation sets
    based on date (most recent reviews go to validation set)
    
    Args:
        df: DataFrame containing the reviews
        valid_percentage: Percentage of reviews to put in validation set (default: 0.2)
    
    Returns:
        train_df: DataFrame with training data
        valid_df: DataFrame with validation data
    """
    # Ensure date column is datetime type
    df = df.copy()
    if not pd.api.types.is_datetime64_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])
    
    # Function to split reviews for a single user
    def split_user_reviews(user_reviews):
        # Sort by date
        user_reviews = user_reviews.sort_values('date')
        
        # Calculate split point
        n_total = len(user_reviews)
        n_valid = max(1, int(n_total * valid_percentage))
        
        # Split the data
        train = user_reviews.iloc[:-n_valid]
        valid = user_reviews.iloc[-n_valid:]
        
        # Mark the split
        train['split'] = 'train'
        valid['split'] = 'valid'
        
        return pd.concat([train, valid])
    
    # Apply the function to each user's reviews
    result_df = df.groupby('UserID').apply(split_user_reviews).reset_index(drop=True)
    
    # Split back into train and valid DataFrames
    train_df = result_df[result_df['split'] == 'train']
    valid_df = result_df[result_df['split'] == 'valid']
    
    # Drop the split column
    train_df = train_df.drop('split', axis=1)
    valid_df = valid_df.drop('split', axis=1)
    
    # Print statistics
    print(f"Total reviews: {len(df)}")
    print(f"Training reviews: {len(train_df)} ({len(train_df)/len(df)*100:.2f}%)")
    print(f"Validation reviews: {len(valid_df)} ({len(valid_df)/len(df)*100:.2f}%)")
    print(f"Number of users: {df['UserID'].nunique()}")
    
    return train_df, valid_df

# Apply the split function to the filtered reviews
train_reviews, valid_reviews = create_train_valid_split(combined_dataset)

In [ ]:
valid_reviews

In [ ]:
import pandas as pd
import numpy as np

# Shuffle the datasets
train_reviews_shuffled = train_reviews.sample(frac=1, random_state=42).reset_index(drop=True)
valid_reviews_shuffled = valid_reviews.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to CSV files
train_reviews_shuffled.to_csv('train_reviews_32k.csv',sep='\t', index=False)
valid_reviews_shuffled.to_csv('valid_reviews_32k.csv',sep='\t', index=False)

print("Shuffled datasets saved to CSV files:")
print(f"- Training set: train_reviews_shuffled.csv ({len(train_reviews_shuffled)} reviews)")
print(f"- Validation set: valid_reviews_shuffled.csv ({len(valid_reviews_shuffled)} reviews)")